# F1 Fantasy 2025: Round-by-Round Optimal Teams

This notebook generates the expected points (EV) and the mathematically optimal team configuration for every round of the 2025 season using the XGBoost model and Knapsack optimization.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
import os

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.model import F1FantasyPredictor, optimize_team
import src.predict_ev as ev_engine


In [2]:
def get_optimal_team_summary(year, round_num, budget=100.0):
    # 1. Load Data
    d_df = pd.read_csv("../data/processed_fantasy_drivers.csv")
    c_df = pd.read_csv("../data/processed_fantasy_constructors.csv")
    
    # 2. Predict Drivers
    d_train = d_df[(d_df["year"] < year) | ((d_df["year"] == year) & (d_df["round"] < round_num))]
    d_predictor = F1FantasyPredictor(asset_type="driver")
    if not d_train.empty:
        d_predictor.train(d_train)
    d_preds = d_predictor.predict_next(d_df, year, round_num)
    
    # 3. Predict Constructors
    c_train = c_df[(c_df["year"] < year) | ((c_df["year"] == year) & (c_df["round"] < round_num))]
    c_predictor = F1FantasyPredictor(asset_type="constructor")
    if not c_train.empty:
        c_predictor.train(c_train)
    c_preds = c_predictor.predict_next(c_df, year, round_num)
    
    all_preds = pd.concat([d_preds, c_preds])
    
    # 4. Optimize
    result = optimize_team(all_preds, budget=budget)
    if not result:
        return pd.DataFrame({"Error": ["No valid team found"]})
    
    summary_data = []
    for i in range(5):
        summary_data.append({"Item": f"Driver {i+1}", "Selection": result["drivers"][i]})
    for i in range(2):
        summary_data.append({"Item": f"Constructor {i+1}", "Selection": result["constructors"][i]})
    
    summary_df = pd.DataFrame(summary_data)
    extra_rows = pd.DataFrame([
        {"Item": "Total Cost", "Selection": f"${result['total_cost']:.1f}M"},
        {"Item": "Predicted Points", "Selection": f"{result['predicted_points']:.2f}"}
    ])
    return pd.concat([summary_df, extra_rows], ignore_index=True)


## 2025 Season Analysis

Iterating through all available rounds in the 2025 dataset.

In [3]:
df_main = pd.read_csv("../data/processed_fantasy.csv")
rounds_2025 = sorted(df_main[df_main["year"] == 2025]["round"].unique())

round_summaries = {}

for r in rounds_2025:
    print(f"Analyzing Round {r}...")
    summary = get_optimal_team_summary(2025, r, budget=100.0)
    if summary is not None:
        round_summaries[r] = summary

print("Generation Complete.")

Analyzing Round 1...
Model (driver) trained. Features: 45. Test RMSE: 14.46


KeyError: "['drivers_predicted_points_sum'] not in index"

## Summary Tables

Below are the optimal configurations for each round.

In [ ]:
from IPython.display import display, HTML

for r, df in round_summaries.items():
    display(HTML(f"<h3>Round {r}</h3>"))
    display(df)

In [ ]:
## Top 5 Teams for Round 1
ev_file_r1 = "../data/ev_reports/ev_report_2025_R1.csv"
top_5_teams = solve_knapsack(ev_file_r1, budget=100.0, top_n=5)

top_5_summaries = []
for idx, team in enumerate(top_5_teams):
    summary_data = []
    for i in range(5):
        summary_data.append({"Item": f"Driver {i+1}", "Selection": team["drivers"][i]})
    for i in range(2):
        summary_data.append({"Item": f"Constructor {i+1}", "Selection": team["constructors"][i]})
    
    summary_df = pd.DataFrame(summary_data)
    extra_rows = pd.DataFrame([
        {"Item": "Total Cost", "Selection": f"${team['total_cost']:.1f}M"},
        {"Item": "Predicted Points", "Selection": f"{team['predicted_points']:.2f}"}
    ])
    summary_df = pd.concat([summary_df, extra_rows], ignore_index=True)
    top_5_summaries.append(summary_df)

print("Top 5 Optimal Teams for 2025 Round 1:")
for i, df in enumerate(top_5_summaries):
    display(HTML(f"<h4>Rank {i+1}</h4>"))
    display(df)